# Step 4a: Training the Two Models

Two parameter-matched regressors share an identical convolutional stem and an identical regression head and differ only in the body between them, three residual convolutional blocks against three Transformer encoder layers. Both are defined in `models.py`. Stem, head, parameter budget, data and training recipe are held fixed, so a difference in behaviour is a property of the body. The bodies themselves differ in more than one respect (convolution against attention, batch against layer normalisation, global average against mean pooling), and no result of this study is attributed to one of those components in isolation.

**One seed per run.** `SEED` is the only value that changes between the five runs. Run identifiers, checkpoints and result files are all derived from it, so the outputs of different seeds never collide. Either edit the parameter cell below, or run the notebook non-interactively, for example with papermill:

```
for s in 0 1 2 3 4; do papermill 04a_Train_Baseline.ipynb runs/04a_seed$s.ipynb -p SEED $s; done
```

The paper uses seeds 0 to 4, each trained on one NVIDIA T4 GPU.

**Note on the convolutional stem.** The spectra reaching this model live on a logarithmic wavelength grid, on which a fixed-width kernel spans a constant *velocity* interval at every wavelength (77.0 km/s per pixel). Since the instrumental resolution element is also a constant velocity width, a single learned filter matches the same line shape across the entire range. On a linear grid the same kernel would span 954 km/s at the blue end and 333 km/s at the red end, and the filters would have to re-learn line shapes region by region.

In [ ]:
# parameters: the training seed of this run (0 to 4 in the paper)
SEED = 0

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import h5py

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR

from common import seed_everything

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {device}")
print(f"Seed for this run  : {SEED}")

## 4.1 Configuration

The Kelvin conversion factor is read from `preprocessing_meta.json` rather than written into the code. Every RMSE reported in physical units is the normalised RMSE multiplied by that factor, so an incorrect constant would rescale every number in the paper by the same silent factor.

In [ ]:
class CFG:
    # Environment
    num_workers = 0

    # Data
    batch_size = 256
    seq_len = 4096

    # Training
    epochs = 100
    learning_rate = 5e-4
    weight_decay = 1e-4
    warmup_epochs = 10
    patience_limit = 15        # early stopping, on validation RMSE
    plateau_patience = 5       # ReduceLROnPlateau, on validation loss
    plateau_factor = 0.5
    min_lr = 1e-6
    grad_clip = 1.0
    huber_delta = 1.0

    # Paths
    data_dir = "data"
    tensors_path = os.path.join(data_dir, "sdss_tensors.h5")
    catalog_path = os.path.join(data_dir, "sdss_final_catalog.csv")
    meta_path = os.path.join(data_dir, "preprocessing_meta.json")

    # Outputs
    output_dir = ""
    results_dir = os.path.join(output_dir, "results")
    ckpt_dir = os.path.join(output_dir, "checkpoints")

    # Optional: also pack results/ into <archive_dir>/results_seed<SEED>.zip,
    # convenient on hosted notebooks where only one folder can be downloaded.
    archive_dir = None

os.makedirs(CFG.results_dir, exist_ok=True)
os.makedirs(CFG.ckpt_dir, exist_ok=True)

# --- Kelvin conversion factor, from Step 2 -------------------------------------
with open(CFG.meta_path) as f:
    prep_meta = json.load(f)

CFG.teff_min = prep_meta['teff_min']
CFG.teff_max = prep_meta['teff_max']
CFG.teff_range = prep_meta['teff_range']

print("Configuration loaded.")
print(f"  Teff scaling : [{CFG.teff_min:.4f}, {CFG.teff_max:.4f}] K")
print(f"  teff_range   : {CFG.teff_range:.4f} K   <- normalised RMSE x this = Kelvin")
print(f"  results -> {CFG.results_dir}")
print(f"  ckpts   -> {CFG.ckpt_dir}")

## 4.2 Parameter-matched architectures

`SharedStem` compresses (1, 4096) to (128, 128). `MaxPool1d` is used rather than average pooling so that isolated high-amplitude pixels survive the reduction. The spike perturbation of Step 5.1 must reach the model body to be measurable, and average pooling would attenuate it before either branch could respond.

The two bodies are sized to match: three residual convolution blocks at 128 channels against three Transformer layers at `d_model=128` with `dim_feedforward=384`. Both terminate in the same `Linear(128, 64) - GELU - Linear(64, 1)` head. Conv-Pool applies channel-wise dropout (`Dropout1d`, p = 0.1) to the residual branch of each block; Conv-Attn uses the standard Transformer dropout of 0.1 on attention and feed-forward outputs.

The definitions live in `models.py` and are imported by every later notebook, so the architecture is written down exactly once. The cell below asserts the parameter counts reported in the paper.

In [ ]:
from models import StellarConvPool, StellarConvAttn, check_parameter_parity

check_parameter_parity();

## 4.3 Data loader

Weights come from the `lds_weight` column written in Step 3, where they were normalised to a mean of exactly 1.0 over the training partition and fixed at 1.0 for validation and test. The loader asserts that property rather than silently re-normalising, so a mismatch between the catalog and the tensor surfaces as an error instead of being absorbed.

In [ ]:
class SDSSDataset(Dataset):
    def __init__(self, h5_path, indices, catalog_df, name=""):
        print(f"Loading {len(indices):,} spectra into RAM ({name})...")

        # sorted for sequential HDF5 reads; row i of the tensor stays aligned
        # with row i of the catalog because Step 2 reset the catalog index
        sorted_indices = np.sort(indices)
        self.catalog_indices = sorted_indices

        w = catalog_df.iloc[sorted_indices]['lds_weight'].values.astype(np.float32)
        self.weights = w

        with h5py.File(h5_path, 'r') as f:
            self.x_data = f['X'][sorted_indices]
            y_raw = f['y_raw'][sorted_indices].astype(np.float64)
        self.y_data = ((y_raw - CFG.teff_min) / CFG.teff_range).astype(np.float32)

        print(f"  weights: mean {w.mean():.6f}, range [{w.min():.4f}, {w.max():.4f}]")
        print(f"  X {self.x_data.shape}, y {self.y_data.shape}")
        print(f"  y range: [{self.y_data.min():.4f}, {self.y_data.max():.4f}]")

    def __len__(self):
        return len(self.weights)

    def __getitem__(self, idx):
        x = torch.tensor(self.x_data[idx], dtype=torch.float32).view(1, -1)
        y = torch.tensor(self.y_data[idx], dtype=torch.float32).squeeze()
        w = torch.tensor(self.weights[idx], dtype=torch.float32)
        return x, y, w


def get_dataloaders(cfg, seed):
    print("Loading catalog...")
    catalog_df = pd.read_csv(cfg.catalog_path)
    assert 'lds_weight' in catalog_df.columns, "run Step 3 first: lds_weight is missing"

    train_idx = catalog_df.index[catalog_df['dataset_split'] == 'train'].values
    val_idx = catalog_df.index[catalog_df['dataset_split'] == 'val'].values
    test_idx = catalog_df.index[catalog_df['dataset_split'] == 'test'].values

    # the weighting scheme of Step 3 must survive intact to this point
    w_train = catalog_df.loc[train_idx, 'lds_weight'].values
    w_other = catalog_df.loc[np.concatenate([val_idx, test_idx]), 'lds_weight'].values
    assert np.isclose(w_train.mean(), 1.0, atol=1e-4), \
        f"training weights have mean {w_train.mean():.6f}, expected 1.0"
    assert np.allclose(w_other, 1.0), "validation/test weights must all equal 1.0"

    train_ds = SDSSDataset(cfg.tensors_path, train_idx, catalog_df, "train")
    val_ds = SDSSDataset(cfg.tensors_path, val_idx,   catalog_df, "val")
    test_ds = SDSSDataset(cfg.tensors_path, test_idx,  catalog_df, "test")

    def worker_init_fn(worker_id):
        np.random.seed(seed + worker_id)

    common = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=True)
    train_loader = DataLoader(train_ds, shuffle=True, worker_init_fn=worker_init_fn, **common)
    val_loader = DataLoader(val_ds, shuffle=False, **common)
    test_loader = DataLoader(test_ds, shuffle=False, **common)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_dataloaders(CFG, SEED)
print(f"\nBatches: train {len(train_loader)}, val {len(val_loader)}, "
      f"test {len(test_loader)}")

## 4.4 Training engine

**Learning-rate schedule.** Linear warm-up over the first `warmup_epochs`, stepped per batch, followed by `ReduceLROnPlateau` on the validation loss, stepped once per epoch. A cosine schedule was rejected because its horizon must be declared in advance while early stopping cuts the run short; the learning rate would then still sit near its peak when training ends.

**Evaluation runs in `eval()` mode** throughout, so batch-normalisation uses its running statistics and dropout is disabled. This is what makes a repeated evaluation of the same input reproducible, and it is a precondition for the perturbation curves of Step 5.

**Reported errors are unweighted.** The LDS weights enter the training loss only. Validation and test samples carry weight 1.0, so the RMSE is a physical error and remains comparable across temperature regimes.

In [ ]:
from tqdm.auto import tqdm
import time

# ---------------------------------------------------------
# LDS-weighted Huber loss
# ---------------------------------------------------------
def lds_huber_loss(preds, targets, weights, delta=1.0):
    preds = preds.view(-1)
    targets = targets.view(-1)
    weights = weights.view(-1)
    loss = nn.functional.huber_loss(preds, targets, reduction='none', delta=delta)
    return (loss * weights).mean()


def train_one_epoch(model, loader, optimizer, warmup_scheduler, current_epoch, cfg, device):
    model.train()
    total_loss = 0.0
    for x, y, w in tqdm(loader, desc="Training", leave=False):
        x, y, w = x.to(device), y.to(device), w.to(device)
        optimizer.zero_grad()
        preds = model(x)
        loss = lds_huber_loss(preds, y, w, delta=cfg.huber_delta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip)
        optimizer.step()
        if current_epoch <= cfg.warmup_epochs:
            warmup_scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, cfg, device, return_preds=False):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []
    for x, y, w in loader:
        x, y, w = x.to(device), y.to(device), w.to(device)
        preds = model(x)
        total_loss += lds_huber_loss(preds, y, w, delta=cfg.huber_delta).item()
        all_preds.append(preds.detach().cpu().numpy().ravel())
        all_targets.append(y.detach().cpu().numpy().ravel())

    all_preds = np.concatenate(all_preds).astype(np.float64)
    all_targets = np.concatenate(all_targets).astype(np.float64)
    rmse = float(np.sqrt(np.mean((all_preds - all_targets) ** 2)))

    if return_preds:
        return total_loss / len(loader), rmse, all_preds, all_targets
    return total_loss / len(loader), rmse


# ---------------------------------------------------------
# Training loop
# ---------------------------------------------------------
def fit_model(model, model_name, seed, train_loader, val_loader, cfg, device):
    run_id = f"{model_name}_seed{seed}"
    print(f"\n{'='*62}\nTraining {run_id}\n{'='*62}")
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")

    optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate,
                            weight_decay=cfg.weight_decay)
    warmup_steps = len(train_loader) * cfg.warmup_epochs
    scheduler_warmup = LinearLR(optimizer, start_factor=0.01,
                                total_iters=warmup_steps)
    scheduler_plateau = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=cfg.plateau_factor,
        patience=cfg.plateau_patience, min_lr=cfg.min_lr
    )

    best_rmse, best_epoch = float('inf'), -1
    patience_counter, early_stopped = 0, False
    ckpt_path = os.path.join(cfg.ckpt_dir, f"{run_id}_best.pth")

    history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_rmse': [],
               'val_rmse_kelvin': [], 'lr': [], 'epoch_time_s': [], 'is_best': []}

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer,
                                     scheduler_warmup, epoch, cfg, device)
        val_loss, val_rmse = evaluate(model, val_loader, cfg, device)

        if epoch > cfg.warmup_epochs:
            scheduler_plateau.step(val_loss)

        epoch_time = time.time() - t0
        current_lr = optimizer.param_groups[0]['lr']
        val_rmse_k = val_rmse * cfg.teff_range

        # single checkpoint / patience block
        is_best = val_rmse < best_rmse
        if is_best:
            best_rmse, best_epoch = val_rmse, epoch
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_rmse'].append(val_rmse)
        history['val_rmse_kelvin'].append(val_rmse_k)
        history['lr'].append(current_lr)
        history['epoch_time_s'].append(epoch_time)
        history['is_best'].append(bool(is_best))

        print(f"Epoch {epoch:03d} [{epoch_time:.0f}s] | LR {current_lr:.2e} | "
              f"Train {train_loss:.5f} | Val {val_loss:.5f} | "
              f"RMSE {val_rmse_k:.1f} K{'  *' if is_best else ''}")

        if patience_counter >= cfg.patience_limit:
            early_stopped = True
            print(f"Early stopping at epoch {epoch}. "
                  f"Best {best_rmse * cfg.teff_range:.1f} K at epoch {best_epoch}.")
            break

    hist_df = pd.DataFrame(history)
    hist_df.insert(0, 'model', model_name)
    hist_df.insert(1, 'seed', seed)
    csv_path = os.path.join(cfg.results_dir, f"history_{run_id}.csv")
    hist_df.to_csv(csv_path, index=False)
    print(f"[saved] {csv_path}")

    meta = {
        'run_id': run_id, 'model': model_name, 'seed': int(seed),
        'n_params': int(n_params),
        'epochs_run': len(history['epoch']),
        'early_stopped': early_stopped,
        'best_epoch': int(best_epoch),
        'best_val_rmse_norm': float(best_rmse),
        'best_val_rmse_kelvin': float(best_rmse * cfg.teff_range),
        'mean_epoch_time_s': float(np.mean(history['epoch_time_s'])),
        'total_time_s': float(np.sum(history['epoch_time_s'])),
        'checkpoint_path': ckpt_path,
    }
    return history, meta


# ---------------------------------------------------------
# Final evaluation on the held-out test partition
# ---------------------------------------------------------
def export_test_results(model, model_name, seed, test_loader, meta, cfg, device):
    run_id = f"{model_name}_seed{seed}"

    # reload the best checkpoint: the weights currently in memory are those of
    # the last epoch, which is not the best one when early stopping fired
    state = torch.load(meta['checkpoint_path'], map_location=device)
    model.load_state_dict(state)
    model = model.to(device)

    _, _, preds_norm, targets_norm = evaluate(
        model, test_loader, cfg, device, return_preds=True)

    # full inverse transform, so the stored values are absolute temperatures
    preds_k = preds_norm * cfg.teff_range + cfg.teff_min
    targets_k = targets_norm * cfg.teff_range + cfg.teff_min
    residual = preds_k - targets_k

    rmse_k = float(np.sqrt(np.mean(residual ** 2)))
    mae_k = float(np.mean(np.abs(residual)))
    bias_k = float(np.mean(residual))
    r2 = float(1.0 - np.sum(residual ** 2) / np.sum((targets_k - targets_k.mean()) ** 2))

    meta.update({'test_rmse_kelvin': rmse_k, 'test_mae_kelvin': mae_k,
                 'test_bias_kelvin': bias_k, 'test_r2': r2,
                 'n_test': int(len(targets_k))})

    # --- error by temperature regime -----------------------------------------
    edges = [3500, 4000, 4500, 5000, 5500, 6000, 6500, 7000, 8000, 9000]
    regime = pd.cut(targets_k, bins=edges)
    per_regime = pd.DataFrame({'true': targets_k, 'resid': residual}) \
        .groupby(regime, observed=True)['resid'] \
        .agg(n='size', rmse=lambda s: float(np.sqrt(np.mean(s ** 2))), bias='mean')
    meta['per_regime'] = {str(k): {'n': int(v['n']), 'rmse_K': float(v['rmse']), 'bias_K': float(v['bias'])}
                          for k, v in per_regime.iterrows()}

    npz_path = os.path.join(cfg.results_dir, f"testpreds_{run_id}.npz")
    np.savez_compressed(
        npz_path,
        y_true_kelvin=targets_k.astype(np.float32),
        y_pred_kelvin=preds_k.astype(np.float32),
        catalog_index=test_loader.dataset.catalog_indices.astype(np.int64),
    )

    json_path = os.path.join(cfg.results_dir, f"summary_{run_id}.json")
    with open(json_path, 'w') as f:
        json.dump(meta, f, indent=2)

    print(f"\nTEST | RMSE {rmse_k:.2f} K | MAE {mae_k:.2f} K | "
          f"Bias {bias_k:+.2f} K | R2 {r2:.4f}")
    print("\nError by temperature regime:")
    display(per_regime.round(2))
    print(f"[saved] {npz_path}")
    print(f"[saved] {json_path}")
    return meta

## 4.5 Convergence plotting

Defined once and reused for both models. The learning rate is overlaid on the error panel so that a drop in the schedule can be read against the response in validation error.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'stix',
})

def plot_training_history(history, model_name, cfg):
    epochs = history['epoch']
    fig, ax = plt.subplots(1, 2, figsize=(16, 5.5), dpi=150)

    ax[0].plot(epochs, history['train_loss'], label='Train', color='#1f77b4', linewidth=2)
    ax[0].plot(epochs, history['val_loss'], label='Validation', color='#ff7f0e', linewidth=2, linestyle='--')
    ax[0].set_yscale('log')
    ax[0].set_title(f"{model_name} — LDS-weighted Huber loss", fontsize=17, fontweight='bold')
    ax[0].set_xlabel("Epoch", fontsize=15)
    ax[0].set_ylabel("Loss", fontsize=15)
    ax[0].tick_params(labelsize=13)
    ax[0].legend(fontsize=13)
    ax[0].grid(True, which="both", ls="--", alpha=0.4)

    rmse_k = np.array(history['val_rmse_kelvin'])
    best_i = int(np.argmin(rmse_k))
    ax[1].plot(epochs, rmse_k, color='#d62728', linewidth=2.2, label='Validation RMSE')
    ax[1].axhline(rmse_k[best_i], color='gray', linestyle=':', alpha=0.85, label=f"Best {rmse_k[best_i]:.1f} K (epoch {epochs[best_i]})")
    ax[1].scatter([epochs[best_i]], [rmse_k[best_i]], color='#d62728', zorder=5, s=45)
    ax[1].set_yscale('log')
    ax[1].set_title(f"{model_name} — physical temperature error", fontsize=17, fontweight='bold')
    ax[1].set_xlabel("Epoch", fontsize=15)
    ax[1].set_ylabel("RMSE [K]", fontsize=15)
    ax[1].tick_params(labelsize=13)
    ax[1].grid(True, which="both", ls="--", alpha=0.4)

    ax2 = ax[1].twinx()
    ax2.plot(epochs, history['lr'], color='#2ca02c', linewidth=1.2, alpha=0.6, label='Learning rate')
    ax2.set_yscale('log')
    ax2.set_ylabel("Learning rate", fontsize=13, color='#2ca02c')
    ax2.tick_params(axis='y', labelcolor='#2ca02c', labelsize=11)
    ax2.grid(False)

    h1, l1 = ax[1].get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax[1].legend(h1 + h2, l1 + l2, fontsize=12, loc='upper right')

    plt.tight_layout()
    plt.show()

## 4.6 Baseline execution

Both models are seeded identically before construction, so their stems start from the same initialisation. The checkpoints written here are the inputs to the perturbation experiments of Step 5.

In [ ]:
# ---------------------------------------------------------
# 1. Conv-Pool
# ---------------------------------------------------------
seed_everything(SEED)
model_pool = StellarConvPool()

history_pool, meta_pool = fit_model(
    model=model_pool, model_name="Conv-Pool", seed=SEED,
    train_loader=train_loader, val_loader=val_loader,
    cfg=CFG, device=device,
)

meta_pool = export_test_results(
    model_pool, "Conv-Pool", SEED, test_loader, meta_pool, CFG, device,
)

plot_training_history(history_pool, "Conv-Pool", CFG)

In [ ]:
# ---------------------------------------------------------
# 2. Conv-Attn
# ---------------------------------------------------------
seed_everything(SEED)
model_attn = StellarConvAttn()

history_attn, meta_attn = fit_model(
    model=model_attn, model_name="Conv-Attn", seed=SEED,
    train_loader=train_loader, val_loader=val_loader,
    cfg=CFG, device=device,
)

meta_attn = export_test_results(
    model_attn, "Conv-Attn", SEED, test_loader, meta_attn, CFG, device,
)

plot_training_history(history_attn, "Conv-Attn", CFG)

## 4.7 Run summary and export

Everything needed to reproduce the figures of the paper without re-training is written under `results/`: the per-epoch history, the summary metrics, and the test-set predictions with their catalog indices. The checkpoints stay in a separate directory, since they are needed for Step 5 but not for plotting.

In [ ]:
import shutil

summary_df = pd.DataFrame([{
    'model': m['model'],
    'seed': m['seed'],
    'n_params': m['n_params'],
    'epochs': m['epochs_run'],
    'early_stop': m['early_stopped'],
    'best_epoch': m['best_epoch'],
    'val_RMSE_K': round(m['best_val_rmse_kelvin'], 2),
    'test_RMSE_K': round(m['test_rmse_kelvin'], 2),
    'test_MAE_K': round(m['test_mae_kelvin'], 2),
    'test_Bias_K': round(m['test_bias_kelvin'], 2),
    'test_R2': round(m['test_r2'], 4),
    'min_per_ep': round(m['mean_epoch_time_s'] / 60, 2),
} for m in [meta_pool, meta_attn]])

print(f"=== SEED {SEED} ===")
display(summary_df)
summary_df.to_csv(os.path.join(CFG.results_dir, f"SUMMARY_seed{SEED}.csv"), index=False)

if CFG.archive_dir:
    archive = os.path.join(CFG.archive_dir, f"results_seed{SEED}")
    shutil.make_archive(archive, 'zip', CFG.results_dir)
    print(f"\nArchive: {archive}.zip")
print("Checkpoints for Step 5 are in:", CFG.ckpt_dir)